In [ ]:
import os
import re
import io
import email
import time
import json
import torch
import shutil
import imaplib
import smtplib
import tempfile
import pandas as pd
import fitz  # PyMuPDF
from PIL import Image
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from email.header import decode_header
from email.mime.text import MIMEText
from hijri_converter import Hijri
# LangChain & Transformers
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever, ParentDocumentRetriever
from langchain.storage import InMemoryStore
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from googleapiclient.discovery import build
from google.oauth2 import service_account
from ArabicOcr import arabicocr  # تأكد من تثبيتها
import gc  # <--- إضافة مهمة جداً لتنظيف الذاكرة
import openpyxl  # <--- ضروري جداً للتعامل مع الإكسل الضخم
# ==========================================
# الإعدادات العامة (Configurations)
# ==========================================
MODEL_ID = "Qwen/Qwen3-32B"
#MODEL_ID = "Qwen/Qwen2.5-32B-Instruct" # تم التغيير لـ 2.5 لضمان العمل، أو استبدله بـ Qwen3 إذا كنت متأكداً من وجوده
GMAIL_EMAIL = "eng.mansour.issa@gmail.com"
APP_PASSWORD = "sugtcfmplficwqzr"
CALENDAR_ID = "eng.mansour.issa@gmail.com"
SERVICE_ACCOUNT_FILE = 'kaust-481121-7ec069937b0c.json'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SCOPES = ['https://www.googleapis.com/auth/calendar']
IMAP_SERVER = "imap.gmail.com"
SMTP_SERVER = "smtp.gmail.com"

# تحميل نموذج التوليد (LLM) عالمياً لاستخدامه في تحليل النية والرد
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
model = torch.compile(model)
llm_pipeline = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=1024,
    # الإضافات الجديدة:
    do_sample=True  ,    # تفعيل أخذ العينات لاستخدام درجة الحرارة
    temperature=0.1,     # خفض الحرارة لزيادة الدقة (0.1 إلى 0.3 قيم ممتازة للالتزام)
    top_p=0.90,           # التركيز على الكلمات الأكثر احتمالية فقط
    repetition_penalty=1.15 # لمنع التكرار غير المفيد في الردود الرسمية
)
#llm_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=1024)

# ==========================================
# القسم الأول: معالجة الملفات (Static Knowledge) بأسلوب Small-to-Parent
# ==========================================
class FileProcessor:
    """كلاس متخصص لمعالجة كافة أنواع الملفات واستخراج النصوص"""
    
    def process_pdf(self, file_path):
        full_text = []
        doc = fitz.open(file_path)
        for i, page in enumerate(doc):
            pix = page.get_pixmap(dpi=300)
            img_path = f"temp_{i}.png"
            pix.save(img_path)
            results = arabicocr.arabic_ocr(img_path, f"out_{i}.jpg")
            structured_text = self._format_smart_text(results)
            full_text.append(structured_text + "\n\n")
            if os.path.exists(img_path): os.remove(img_path)
            if os.path.exists(f"out_{i}.jpg"): os.remove(f"out_{i}.jpg")
            # تنظيف الذاكرة بعد كل صفحة
            del pix
            gc.collect()
        doc.close()
        return "".join(full_text)

    def _format_smart_text(self, results):
        if not results: return ""
        boxes = [{'y': r[0][0][1], 'x_left': r[0][0][0], 'x_right': r[0][1][0], 'text': r[1], 'h': r[0][2][1]-r[0][0][1]} for r in results]
        boxes.sort(key=lambda b: b['y'])
        rows, current_row = [], [boxes[0]]
        for i in range(1, len(boxes)):
            if abs(boxes[i]['y'] - current_row[-1]['y']) < (current_row[-1]['h'] / 2):
                current_row.append(boxes[i])
            else:
                rows.append(current_row)
                current_row = [boxes[i]]
        rows.append(current_row)
        lines = []
        for row in rows:
            row.sort(key=lambda b: b['x_right'], reverse=True)
            line_content = ""
            for j in range(len(row)):
                line_content += row[j]['text']
                if j < len(row) - 1:
                    gap = row[j]['x_left'] - row[j+1]['x_right']
                    if gap > (row[j]['h'] * 2):
                        line_content += " | "
                    else:
                        line_content += " "
            lines.append(line_content)
        return "\n".join(lines)

    def process_text(self, path):
        """قراءة الملفات النصية بترميز UTF-8"""
        try:
            with open(path, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e:
            print(f"⚠ خطأ في قراءة الملف النصي {path}: {e}")
            return ""

    def process_docx(self, path):
        """معالجة ملفات الوورد"""
        from docx import Document as DocxDoc
        try:
            doc = DocxDoc(path)
            content = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
            for table in doc.tables:
                for row in table.rows:
                    row_text = " | ".join([cell.text.strip() for cell in row.cells])
                    content.append(row_text)
            return "\n".join(content)
        except Exception as e:
            print(f"⚠ خطأ في معالجة ملف الوورد {path}: {e}")
            return ""

    def process_excel(self, path):
        """
        معالجة ملفات الإكسل الضخمة باستخدام openpyxl في وضع القراءة فقط.
        هذا يتجاوز خطأ chunksize ويوفر استهلاك الذاكرة بشكل كبير.
        """
        try:
            # data_only=True لجلب القيم وليس المعادلات
            # read_only=True هو المفتاح لمنع تحميل الملف كاملاً في الرام
            wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
            
            all_sheets_text = []
            
            for sheet_name in wb.sheetnames:
                try:
                    sheet = wb[sheet_name]
                    sheet_rows_buffer = [f"Sheet: {sheet_name}"]
                    row_counter = 0
                    
                    # iter_rows يقوم بجلب البيانات سطر بسطر (Streaming)
                    for row in sheet.iter_rows(values_only=True):
                        # تنظيف الصف واختيار الخلايا غير الفارغة
                        clean_row = [str(cell).strip() for cell in row if cell is not None and str(cell).strip() != ""]
                        
                        if clean_row:
                            sheet_rows_buffer.append(" | ".join(clean_row))
                        
                        row_counter += 1
                        
                        # كل 1000 سطر، قم بتفريغ القائمة النصية وتنظيف الذاكرة
                        if row_counter >= 1000:
                            all_sheets_text.append("\n".join(sheet_rows_buffer))
                            sheet_rows_buffer = [] # إعادة التعيين
                            row_counter = 0
                            gc.collect()
                    
                    # إضافة ما تبقى في المخزن المؤقت
                    if sheet_rows_buffer:
                        all_sheets_text.append("\n".join(sheet_rows_buffer))
                    
                    # تنظيف بعد انتهاء الشيت
                    del sheet_rows_buffer
                    gc.collect()
                    
                except Exception as sheet_error:
                    print(f"⚠ تحذير في الشيت {sheet_name}: {sheet_error}")
                    continue

            wb.close()
            final_text = "\n\n".join(all_sheets_text)
            
            del all_sheets_text
            gc.collect()
            
            return final_text

        except Exception as e:
            print(f"⚠ خطأ في ملف الإكسل {path}: {e}")
            return "تعذر استخراج النص."






def find_parent_large_chunks(small_docs, large_docs):
    """البحث عن القطع الكبيرة التي تحتوي على نص القطع الصغيرة المسترجعة"""
    parent_chunks = []
    unique_content_set = set()
    for small_doc in small_docs:
        for large_doc in large_docs:
            if small_doc.page_content in large_doc.page_content:
                if large_doc.page_content not in unique_content_set:
                    parent_chunks.append(large_doc)
                    unique_content_set.add(large_doc.page_content)
                break
    return parent_chunks


    
# ==========================================
# القسم المحدث: بناء القاعدة المعرفية بنظام (Pattern + Small-to-Parent)
# ==========================================
def initialize_fixed_knowledge(processor, embed_model):
    fixed_files = ["modified_comp_updated3 (1).txt", "Depot_Updated.txt",
                    "private_cond.txt", "email_history.txt"]
    
    initial_docs = []
    for f in fixed_files:
        if os.path.exists(f):
            text = processor.process_text(f)
            initial_docs.append(Document(page_content=text, metadata={'source': f}))

    pattern_split_docs = regex_split_documents(initial_docs)
    text_splitter_large = RecursiveCharacterTextSplitter(chunk_size=1300, chunk_overlap=200)
    text_splitter_small = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=25)
    
    docs_large = []
    docs_small = []
    for doc in pattern_split_docs:
        heading_match = re.match(r"^(اللائحة المادة\s+\d+|النظام المادة\s+\d+|المادة\s+\d+|محضر تسليم|الموقع:|الفقرة\s+\d+|إيميل وارد بتاريخ\s+.*|إيميل صادر بتاريخ\s+.*)", doc.page_content)
        heading = heading_match.group(0) if heading_match else ""
        
        for chunk in text_splitter_large.split_text(doc.page_content):
            docs_large.append(Document(page_content=f"{heading} {chunk}", metadata=doc.metadata))
        
        for chunk in text_splitter_small.split_text(doc.page_content):
            docs_small.append(Document(page_content=f"{heading} {chunk}", metadata=doc.metadata))

    vs_large = FAISS.from_documents(docs_large, embed_model)
    vs_small = FAISS.from_documents(docs_small, embed_model)

    # --- التحديث هنا: رفع قيم K لـ BM25 ---
    bm25_large = BM25Retriever.from_documents(docs_large)
    bm25_large.k = 6  
    
    bm25_small = BM25Retriever.from_documents(docs_small)
    bm25_small.k = 18
    
    print(f"✅ تم بناء القاعدة المعرفية: {len(docs_large)} قطع كبيرة و {len(docs_small)} قطع صغيرة.")
    
    return {
        "vs_large": vs_large,
        "vs_small": vs_small,
        "bm25_large": bm25_large,
        "bm25_small": bm25_small,
        "docs_large": docs_large
    }






# ==========================================
# دالة مساعدة لتقسيم المستندات بناءً على الأنماط (Regex)
# ==========================================
def regex_split_documents(documents):
    """تقسيم النصوص بناءً على أنماط المواد واللوائح لضمان عدم انكسار النص القانوني"""
    pattern = r"(?=(اللائحة المادة\s+\d+\s+من\s+نظام\s+المنافسات\s+والمشتريات\s+الحكومية|النظام المادة\s+\d+\s+من\s+نظام\s+المنافسات\s+والمشتريات\s+الحكومية|المادة\s+\d+\s+من\s+قواعد\s+واجراءات\s+المستودعات\s+الحكومية|محضر تسليم(?:\s+\S+){1,6}|الموقع:|الفقرة\s+\d+|إيميل وارد بتاريخ\s+[\d-]+\s+[\d:]+|إيميل صادر بتاريخ\s+[\d-]+\s+[\d:]+))"
    
    new_documents = []
    for doc in documents:
        content = doc.page_content
        metadata = doc.metadata
        parts = re.split(pattern, content)
        
        current_chunk = ""
        current_heading = ""
        for part in parts:
            if not part or not part.strip(): continue
            part = part.strip()
            
            # التحقق إذا كان الجزء هو عنوان مادة (بشكل صارم)
            is_heading = re.match(r"^(اللائحة المادة\s+\d+|النظام المادة\s+\d+|المادة\s+\d+|محضر تسليم|الموقع:|الفقرة\s+\d+|إيميل وارد بتاريخ\s+.*|إيميل صادر بتاريخ\s+.*)", part)
            
            if is_heading:
                if current_chunk.strip():
                    new_documents.append(Document(page_content=current_chunk.strip(), metadata=metadata))
                current_heading = part
                current_chunk = f"{current_heading} "  # تكرار العنوان داخل المحتوى كما في المرفق
            elif current_chunk:
                current_chunk += part + " "
                
        if current_chunk.strip():
            new_documents.append(Document(page_content=current_chunk.strip(), metadata=metadata))
            
    return new_documents













import json
import re
from datetime import datetime, timedelta
from hijri_converter import Hijri, Gregorian
from googleapiclient.discovery import build
from google.oauth2 import service_account
try:
    credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
    service = build('calendar', 'v3', credentials=credentials)
except NameError:
    # هذا الجزء فقط لتجنب الخطأ إذا نسخت الكود بدون تعريف المتغيرات، في البيئة الحقيقية سيتم استخدام المتغيرات الموجودة
    service = None 

def get_current_context():
    """استخراج السياق الزمني الحالي بدقة مع ضبط التوقيت للسعودية (UTC+3)"""
    # استخدام UTC ثم إضافة 3 ساعات لضمان أن "اليوم" يطابق توقيت السعودية
    now = datetime.utcnow() + timedelta(hours=3)
    
    days_map = {0: "الاثنين", 1: "الثلاثاء", 2: "الأربعاء", 3: "الخميس", 4: "الجمعة", 5: "السبت", 6: "الأحد"}
    day_name = days_map[now.weekday()]
    
    hijri_now = Gregorian(now.year, now.month, now.day).to_hijri()
    return f"اليوم هو {day_name}، التاريخ الميلادي: {now.strftime('%Y-%m-%d')}، التاريخ الهجري: {hijri_now.year}-{hijri_now.month}-{hijri_now.day}، الوقت الحالي: {now.strftime('%H:%M')}"

def arabic_to_english_numbers(s):
    if not s: return ""
    arabic_map = {'٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'}
    return ''.join(arabic_map.get(char, char) for char in str(s))

def calculate_next_weekday(target_day_idx):
    """حساب تاريخ أقرب يوم أسبوع قادم رياضياً بناءً على الفهرس (0-6) بتوقيت السعودية"""
    if target_day_idx is None: return None
    
    # تصحيح الوقت ليكون بتوقيت السعودية (UTC+3)
    # هذا يمنع الخطأ عند الحساب في ساعات الليل المتأخرة أو الفجر
    today_ksa = datetime.utcnow() + timedelta(hours=3)
    current_idx = today_ksa.weekday()
    
    days_ahead = target_day_idx - current_idx
    
    # إذا كان اليوم المطلوب هو نفس يومنا الحالي أو مضى في ترتيب الأسبوع، ننتقل للأسبوع القادم
    if days_ahead <= 0:
        days_ahead += 7
        
    next_date = today_ksa + timedelta(days=days_ahead)
    return next_date.strftime("%Y-%m-%d")

def calculate_next_hijri(day, month, year=None):
    """حساب أقرب تاريخ ميلادي يوافق التاريخ الهجري المطلوب"""
    try:
        # نستخدم توقيت السعودية هنا أيضاً للاتساق
        now = datetime.utcnow() + timedelta(hours=3)
        current_hijri = Gregorian(now.year, now.month, now.day).to_hijri()
        
        if year is None:
            target_year = current_hijri.year
            # إذا كان الشهر واليوم قد مضى في هذه السنة الهجرية، ننتقل للسنة القادمة
            if (month < current_hijri.month) or (month == current_hijri.month and day < current_hijri.day):
                target_year += 1
        else:
            target_year = year
        
        greg_date = Hijri(target_year, month, day).to_gregorian()
        return greg_date.strftime("%Y-%m-%d")
    except Exception as e:
        print(f"Error calculating Hijri date: {e}")
        return None

def process_gregorian_date(date_str, extracted_data):
    """
    معالجة التاريخ الميلادي مع إعطاء الأولوية للتاريخ الصريح (يوم/شهر)
    على حساب حسابات أيام الأسبوع التقريبية.
    """
    if not date_str: return None
    clean_str = arabic_to_english_numbers(date_str).strip()
    # 1. الأولوية القصوى: البناء المباشر من البيانات الرقمية المستخرجة (اليوم والشهر)
    # لأن الـ LLM دقيق في استخراج الأرقام المذكورة، ولكنه قد يخطئ في حساب يوم الأسبوع
    day = extracted_data.get("day")
    month = extracted_data.get("month")
    year = extracted_data.get("year")
    
    # توحيد التوقيت
    now = datetime.utcnow() + timedelta(hours=3)
    
    if day and month:
        try:
            d = int(day)
            m = int(month)
            
            # تحديد السنة
            if year:
                y = int(year)
                # تصحيح السنوات المختصرة (مثال: 25 -> 2025)
                if y < 100: y += 2000
            else:
                # منطق إكمال السنة الناقصة
                y = now.year
                temp_dt = datetime(y, m, d)
                # إذا كان التاريخ في الماضي بالنسبة لليوم، نفترض السنة القادمة
                # مثال: نحن في ديسمبر والطلب "1 يناير"، إذن 1 يناير السنة القادمة
                if temp_dt.date() < now.date():
                    y += 1
            
            final_dt = datetime(y, m, d)
            return final_dt.strftime("%Y-%m-%d")
            
        except ValueError:
            # في حال كانت الأرقام غير منطقية، ننتقل للمحاولات الأخرى
            pass

    # 2. التعامل مع الكلمات الدلالية المباشرة "غدا" أو "اليوم"
    if "غدا" in clean_str.lower() or "غداً" in clean_str:
        return (now + timedelta(days=1)).strftime("%Y-%m-%d")
    if "اليوم" in clean_str.lower():
        return now.strftime("%Y-%m-%d")

    # 3. الملاذ الأخير: التعامل مع أيام الأسبوع (فقط إذا لم يذكر تاريخ رقمي)
    # مثال: "أريد موعد يوم الخميس القادم" (بدون ذكر تاريخ)
    target_day_idx = extracted_data.get("weekday_index")
    if target_day_idx is not None:
        return calculate_next_weekday(target_day_idx)
        
    return None

def process_hijri_date(date_str, extracted_data):
    """
    معالجة التاريخ الهجري بالاعتماد على تحليل LLM.
    """
    if not date_str: return None
    
    # الاعتماد المباشر على ما استخرجه الـ LLM
    day = extracted_data.get("day")
    month = extracted_data.get("month")
    year = extracted_data.get("year")
    
    if day and month:
        return calculate_next_hijri(int(day), int(month), int(year) if year else None)
    
    # لا توجد محاولات Regex يدوية هنا، نعتمد على ذكاء النموذج
    return None

def extract_appointment_info(email_body):
    """
    استخراج النية وبيانات الموعد باستخدام LLM، مع تفصيل اليوم/الشهر/السنة
    وتحويل أسماء الأيام والأشهر إلى أرقام باستخدام الـ prompt
    """
    current_time_context = get_current_context()
    
    system_prompt = f"""
أنت مساعد ذكي متخصص في استخراج بيانات المواعيد بدقة عالية. {current_time_context}.
مهمتك الرئيسية: تحليل النص بدقة لاستخراج النية والبيانات بصيغة JSON فقط.
كن قوياً ودقيقاً في الفهم، وتعامل مع الأخطاء الإملائية، التنسيقات المختلفة، والاختصارات.
افترض أن الطلب يتعلق بجدولة موعد جديد معك أو تأكيد موعد موجود إلا إذا كان واضحاً خلاف ذلك.

قواعد الاستخراج الأساسية:
- أعد النتيجة بصيغة JSON فقط بدون أي شرح أو نص إضافي.
- لا تُهمل أي تاريخ أو وقت مذكور صراحة في النص.
- إذا وُجد أكثر من تاريخ، اختر التاريخ المرتبط بالموعد أو الاجتماع.

قواعد تحديد النية (Intent Rules) - هام جداً:
- "schedule": 
  1. لطلب جدولة موعد جديد.
  2. إذا كان النص يقترح موعداً محدداً ويطلب التأكيد عليه (مثال: "هل نؤكد الموعد يوم كذا؟"، "أقترح يوم كذا"). في هذه الحالة، المستخدم يريد حجز هذا الوقت، لذا اعتبرها "schedule".
- "confirm": 
  1. فقط إذا كان السؤال يستفسر عن حالة موعد محجوز مسبقاً.
  2. لا تستخدم "confirm" إذا كان النص يحتوي على تاريخ ووقت جديدين يُراد تثبيتهما.

2. الفترات الزمنية المبهمة (Vague Times) - إلزامي:
   - إذا ذُكرت فترة بدون ساعة محددة، استخدم الأوقات الافتراضية التالية:
     - "صباحاً" أو "الصباح" أو "بداية الدوام" -> "09:00"
     - "فترة ما بعد الظهر" أو "الظهر" أو "بعد الظهر" -> "13:00"
     - "العصر" -> "16:00"
     - "المساء" أو "ليلاً" أو "في وقت متأخر" -> "19:00"
   - مثال: "يوم الثلاثاء فترة ما بعد الظهر" -> start_time: "13:00".
   
الحقول المطلوبة:
- "intent": 
  - "schedule" لطلب جدولة موعد جديد
  - "confirm" لتأكيد موعد
  - "none" إذا لم يكن النص متعلقاً بموعد
- "date": النص الخام للتاريخ كما ورد في الإيميل (مثال: "2 يناير القادم"، "2025/12/29"، "10 رجب 1447هـ").
- "date_type": 
  - "gregorian" للتاريخ الميلادي
  - "hijri" للتاريخ الهجري
- "day": رقم اليوم (1–31) إذا وجد.
- "month": رقم الشهر (1–12) إذا وجد.
- "year": رقم السنة إذا وجدت.
- "weekday_index": رقم يوم الأسبوع إذا ذُكر.
- "start_time": وقت بدء الموعد بصيغة 24 ساعة (HH:MM).
- "duration": مدة الاجتماع بالدقائق (افتراضي 60 إذا لم تُذكر).
- "title": عنوان مختصر وواضح للموعد (مثل: "اجتماع لمناقشة تعثر مؤسسة").

تحويل الأسماء إلى أرقام (إلزامي):
1) أيام الأسبوع:
- إذا ذُكر يوم أسبوع (مثل: الاثنين، الثلاثاء، الأربعاء، الخميس، الجمعة، السبت، الأحد)
  أعد "weekday_index" حسب الآتي:
  0 = الاثنين
  1 = الثلاثاء
  2 = الأربعاء
  3 = الخميس
  4 = الجمعة
  5 = السبت
  6 = الأحد

2) الأشهر الهجرية:
- إذا ذُكر شهر هجري (حتى مع أخطاء إملائية)، حوّله إلى رقم:
  1 محرم
  2 صفر
  3 ربيع الأول
  4 ربيع الثاني
  5 جمادى الأولى
  6 جمادى الثانية
  7 رجب
  8 شعبان
  9 رمضان
  10 شوال
  11 ذو القعدة
  12 ذو الحجة
- تعامل مع صيغ مثل: "جمادي"، "ذو القعده"، "ربيع الاول".

3) الأشهر الميلادية العربية:
- إذا ذُكر شهر ميلادي عربي، حوّله إلى "month" كرقم:
  1 يناير
  2 فبراير
  3 مارس
  4 أبريل
  5 مايو
  6 يونيو
  7 يوليو
  8 أغسطس
  9 سبتمبر
  10 أكتوبر
  11 نوفمبر
  12 ديسمبر
- تعامل مع صيغ مثل: "اكتوبر"، "ديسيمبر"، "جانفي".

قواعد استكمال التاريخ الناقص:
- إذا ذُكر اليوم والشهر فقط (مثل: "2 يناير"):
  - استخدم السنة الحالية، وإذا كان التاريخ قد مضى فاجعل السنة القادمة.
- إذا ذُكر "القادم":
  - اختر أقرب تاريخ قادم من السياق الزمني الحالي.
- إذا ذُكر يوم أسبوع فقط:
  - أعد "weekday_index" ولا تُخمن اليوم الرقمي.
- إذا ذُكر تاريخ هجري بدون سنة:
  - استخدم السنة الهجرية الحالية أو القادمة حسب السياق.

قواعد الوقت:
- إذا ذُكر "الساعة السادسة مساء":
  - حوّلها إلى "18:00".
- إذا لم يُذكر وقت:
  - استخدم "09:00" افتراضياً.

قواعد التعامل مع التواريخ والأوقات النسبية (مثل "اليوم"، "غداً"، "بعد 10 دقائق"، "بعد ساعة"):
- استخدم السياق الزمني الحالي لحساب التاريخ والوقت المطلق.
- أعد "day"، "month"، "year"، "start_time" كقيم مطلقة محسوبة.
- مثال: إذا كان "بعد 10 دقائق" والوقت الحالي 14:00، فـ "start_time": "14:10"، و"day"/"month"/"year" الحالي.
- مثال: إذا كان "غداً الساعة 9 صباحاً"، فـ "day"/"month"/"year" للغد، "start_time": "09:00".
- لـ "بعد ساعة"، أضف ساعة واحدة إلى الوقت الحالي وحدث التاريخ إذا لزم.
- تأكد من أن النتيجة دائماً في المستقبل أو الحاضر إذا أمكن.

قواعد صارمة لتحويل الوقت (إلزامي بدون استثناء):
- أي وقت يُذكر مع كلمات تدل على المساء يجب تحويله إلى نظام 24 ساعة بإضافة 12 ساعة إذا كان بين 1 و 11.
- كلمات المساء تشمل (على سبيل المثال لا الحصر):
  مساء، مساً، م، PM، بعد الظهر، ظهراً، الظهر، العصر، عصراً، المغرب، ليلاً، الليل.
- أمثلة إلزامية:
  - "1:00 ظهراً" → "13:00"
  - "الساعة 3 العصر" → "15:00"
  - "8 مساءً" → "20:00"
  - "11 ليلاً" → "23:00"
- أي وقت يُذكر مع كلمات تدل على الصباح يُبقى كما هو (1–11).
- كلمات الصباح تشمل:
  صباحاً، صباح، ص، AM، الفجر، فجراً، الضحى.
- أمثلة:
  - "1:00 صباحاً" → "01:00"
  - "9 صباح" → "09:00"
- إذا ذُكرت ساعة فقط بدون دقائق (مثل: "الساعة 1 ظهراً"):
  - افترض الدقائق "00".
  - مثال: "الساعة 1 ظهراً" → "13:00"
- إذا ذُكر وقت رقمي مع كلمة مساء أو ظهراً (مثل "1:00 م"):
  - اعتبره وقتاً مسائياً دائماً.

أعد النتيجة النهائية بصيغة JSON فقط، بدون أي تعليق أو نص إضافي.
"""
    
    prompt = f"حلل هذا النص بدقة: {email_body}"
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]
    
    # محاكاة استدعاء LLM (يفترض وجود tokenizer و llm_pipeline لـ Qwen 32b)
    # تأكد أن المتغيرات tokenizer و llm_pipeline معرفة في النطاق العام
    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        raw_output = llm_pipeline(text_input)[0]['generated_text'].replace(text_input, "").strip()
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except Exception as e:
        print(f"LLM Error: {e}")
        return {"intent": "none"}
    return {"intent": "none"}

def generate_llm_response(context_data):
    system_prompt = "أنت مساعد إداري ذكي. اكتب رداً مختصراً ورسمياً ومهذباً بالعربية واذكر اسم المرسل في البداية ان وجد مثلا السيد محمد .. , واختتم بتقبلو تحياتي مدير إدارة المشاريع"
    user_prompt = f"سياق الرد: {context_data}. اكتب الرد النهائي."
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    
    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return llm_pipeline(text_input)[0]['generated_text'].replace(text_input, "").strip()
    except:
        return "تمت المعالجة."

def handle_calendar_intent(email_body, sender_email):
    """
    المعالج الرئيسي: استخراج النية، حفظ البيانات في JSON، معالجة التاريخ (مع التعامل مع الناقص)، وإدارة التقويم
    """
    
    # طباعة الوقت عند ورود إيميل جديد
    print(f"\n--- [New Email Processing] Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ---")
    
    # 1. استخراج المعلومات باستخدام LLM
    info = extract_appointment_info(email_body)
    intent = info.get("intent", "none")
    
    if intent == "none" or not info.get("date"):
        return None
    
    # 2. حفظ بيانات الموعد في ملف JSON
    json_filename = f"appointment_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(json_filename, 'w', encoding='utf-8') as json_file:
        json.dump(info, json_file, ensure_ascii=False, indent=4)
    print(f"تم حفظ بيانات الموعد في: {json_filename}")
    
    # 3. معالجة التاريخ بناءً على النوع، مع التعامل مع الناقص رياضياً
    target_date = None
    extracted_data = {
        "day": info.get("day"),
        "month": info.get("month"),
        "year": info.get("year"),
        "weekday_index": info.get("weekday_index")
    }
    if info.get("date_type") == "hijri":
        target_date = process_hijri_date(info["date"], extracted_data)
    else:
        target_date = process_gregorian_date(info["date"], extracted_data)
    
    if not target_date:
        return generate_llm_response({"status": "error_date", "original_date": info["date"]})
    
    start_time = arabic_to_english_numbers(info.get("start_time", "09:00"))
    if ":" not in start_time: start_time = f"{start_time.zfill(2)}:00"
    
    try:
        start_dt = datetime.strptime(f"{target_date} {start_time}", "%Y-%m-%d %H:%M")
        start_iso = start_dt.isoformat() + "+03:00"
        end_iso = (start_dt + timedelta(minutes=info.get("duration", 60))).isoformat() + "+03:00"
    except:
        return generate_llm_response({"status": "error_time"})
    
    if intent == "schedule":
        # التأكد من عدم وجود متغير service كـ None
        if service:
            check_body = {"timeMin": start_iso, "timeMax": end_iso, "items": [{"id": CALENDAR_ID}]}
            busy_slots = service.freebusy().query(body=check_body).execute()['calendars'][CALENDAR_ID]['busy']
            
            if not busy_slots:
                event = {
                    'summary': info.get("title", "اجتماع"),
                    'description': f"جدولة آلية من: {sender_email}",
                    'start': {'dateTime': start_iso, 'timeZone': 'Asia/Riyadh'},
                    'end': {'dateTime': end_iso, 'timeZone': 'Asia/Riyadh'},
                }
                service.events().insert(calendarId=CALENDAR_ID, body=event).execute()
                return generate_llm_response({"status": "booked", "date": target_date, "time": start_time, "title": info["title"]})
            else:
                return generate_llm_response({"status": "busy", "date": target_date, "time": start_time})
        else:
            return "عذراً، خدمة التقويم غير متصلة."
            
    elif intent == "confirm":
        if service:
            events_result = service.events().list(
                calendarId=CALENDAR_ID, timeMin=target_date+"T00:00:00+03:00",
                timeMax=target_date+"T23:59:59+03:00", singleEvents=True
            ).execute()
            events = events_result.get('items', [])
            
            if events:
                return generate_llm_response({"status": "confirmed_exists", "date": target_date, "event_title": events[0]['summary']})
            else:
                return generate_llm_response({"status": "not_found", "date": target_date})
        else:
            return "عذراً، خدمة التقويم غير متصلة."
            
    return None

    
# ==========================================
# القسم الثالث: استرجاع الملفات الثابتة (RAG - Parent Retrieval)
# ==========================================
def get_fixed_context(query, kb):
    """استرجاع هجين مكثف: 8 كبيرة و 16 صغيرة من FAISS و BM25 وتحويلها للأصول"""
    
    # 1. الاسترجاع المباشر للقطع الكبيرة (K=8 من كل مصدر)
    faiss_large_docs = kb["vs_large"].as_retriever(search_kwargs={"k": 6}).invoke(query) # تم التحديث إلى 8
    bm25_large_docs = kb["bm25_large"].invoke(query) # ستسترجع 8 بناءً على التهيئة السابقة
    
    # 2. الاسترجاع للقطع الصغيرة (K=16 من كل مصدر)
    faiss_small_docs = kb["vs_small"].as_retriever(search_kwargs={"k": 18}).invoke(query) # تم التأكيد على 16
    bm25_small_docs = kb["bm25_small"].invoke(query) # ستسترجع 16 بناءً على التهيئة السابقة
    
    # دمج نتائج القطع الصغيرة للبحث عن أصولها
    all_small_hits = faiss_small_docs + bm25_small_docs
    
    # 3. تحويل كافة القطع الصغيرة المسترجعة (32 قطعة كحد أقصى) إلى أصولها الكبيرة (Parents)
    parents_from_small = find_parent_large_chunks(all_small_hits, kb["docs_large"])
    
    # 4. دمج كل النتائج (8 كبيرة FAISS + 8 كبيرة BM25 + أصول القطع الصغيرة) مع منع التكرار
    all_context_docs = faiss_large_docs + bm25_large_docs + parents_from_small
    
    unique_content = set()
    final_context_list = []
    
    for doc in all_context_docs:
        if doc.page_content not in unique_content:
            final_context_list.append(doc.page_content)
            unique_content.add(doc.page_content)
            
    return "\n---\n".join(final_context_list)










# ==========================================
# القسم الرابع: استرجاع المرفقات بنظام Small-to-Parent Retrieval
# ==========================================

def get_attachment_context(attachments_list, query, processor, embed_model):
    """
    يعالج المرفقات بتقسيمها لقطع (كبيرة وصغيرة)، يبحث في الصغيرة ويسترجع الكبيرة المقابلة لها.
    """
    if not attachments_list: return ""
    
    docs_large = [] # قائمة لتخزين القطع الكبيرة (الآباء)
    docs_small = [] # قائمة لتخزين القطع الصغيرة (الأبناء)
    mandatory_headers = [] # الرؤوس الأساسية لكل ملف
    
    # 1. إعداد مقسمات النصوص (نفس إعدادات الملفات الثابتة لتوحيد الدقة)
    text_splitter_large = RecursiveCharacterTextSplitter(chunk_size=1400, chunk_overlap=200)
    text_splitter_small = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=25)
    
    for path in attachments_list:
        try:
            text = ""
            # استخراج النص بناءً على نوع الملف عبر القسم الأول
            if path.endswith('.pdf'): text = str(processor.process_pdf(path))
            elif path.endswith(('.xlsx', '.xls')): text = str(processor.process_excel(path))
            elif path.endswith('.docx'): text = processor.process_docx(path)
            elif path.endswith(('.txt', '.note')): text = processor.process_text(path)
            else: continue
            
            if not text or not text.strip(): continue
            
            file_name = os.path.basename(path)
            
            # حفظ "رأس الملف" بشكل إجباري (أول 1000 حرف) لتعريف الموديل بالوثيقة
            mandatory_headers.append(f"--- مقدمة ملف: {file_name} ---\n{text[:1000]}...")
            
            # 2. تقسيم النص إلى قطع كبيرة (الآباء)
            file_large_chunks = []
            for chunk in text_splitter_large.split_text(text):
                file_large_chunks.append(Document(page_content=chunk, metadata={'source': file_name}))
            docs_large.extend(file_large_chunks)
            
            # 3. تقسيم النص إلى قطع صغيرة (الأبناء)
            file_small_chunks = []
            for chunk in text_splitter_small.split_text(text):
                file_small_chunks.append(Document(page_content=chunk, metadata={'source': file_name}))
            docs_small.extend(file_small_chunks)
            
            # تنظيف الذاكرة بعد كل ملف
            del text
            del file_large_chunks
            del file_small_chunks
            gc.collect()

        except Exception as e:
            print(f"⚠ خطأ في معالجة المرفق {path}: {e}")
            gc.collect()

    if not docs_small: return ""
    
    # 4. بناء مخزن متجهات لحظي للقطع الصغيرة (البحث يكون هنا)
    vs_small_attach = FAISS.from_documents(docs_small, embed_model)
    
    # 5. استرجاع أفضل القطع الصغيرة (k=12 لضمان تغطية واسعة)
    small_results = vs_small_attach.as_retriever(search_kwargs={"k": 18}).invoke(query)
    
    # 6. توسيع القطع الصغيرة المسترجعة إلى أصولها الكبيرة (Parents)
    # نستخدم الدالة المساعدة find_parent_large_chunks الموجودة في كودك
    parent_results = find_parent_large_chunks(small_results, docs_large)
    
    # 7. تجميع السياق النهائي (الرؤوس + القطع الكبيرة)
    final_attachment_ctx = []
    final_attachment_ctx.append("### [معلومات أساسية من المرفقات]:")
    final_attachment_ctx.extend(mandatory_headers)
    
    final_attachment_ctx.append("\n### [تفاصيل موسعة مسترجعة من المرفقات]:")
    unique_content = set()
    for doc in parent_results:
        content = f"[{doc.metadata['source']}]: {doc.page_content}"
        if content not in unique_content:
            final_attachment_ctx.append(content)
            unique_content.add(content)
    
    # تنظيف نهائي للكائنات الكبيرة
    del docs_large
    del docs_small
    del vs_small_attach
    gc.collect()
    
    return "\n---\n".join(final_attachment_ctx)

















def safe_decode(payload):
    if payload is None: return ""
    for enc in ["utf-8", "iso-8859-1", "cp1256"]:
        try: return payload.decode(enc)
        except: continue
    return payload.decode("utf-8", errors="ignore")
def html_to_text(html): return BeautifulSoup(html, "html.parser").get_text()
def send_reply(to, subject, text):
    msg = MIMEText(text, "plain", "utf-8")
    msg["Subject"] = f"Re: {subject}"
    msg["From"] = GMAIL_EMAIL
    msg["To"] = to
    try:
        with smtplib.SMTP(SMTP_SERVER, 587, timeout=20) as server:
            server.starttls()
            server.login(GMAIL_EMAIL, APP_PASSWORD)
            server.sendmail(GMAIL_EMAIL, [to], msg.as_string())
        print(f"✔ تم إرسال رد إلى {to}")
    except Exception as e:
        print(f"⚠ خطأ إرسال رد: {e}")








        
def check_new_emails(processor):
    """
    فحص البريد الوارد واستخراج الرسائل الجديدة مع المرفقات.
    تحديث: يعيد نص الإيميل الأصلي فقط (Clean Body) دون دمج نص المرفقات لضمان نظافة التاريخ.
    """
    import gc
    new_emails = []
    
    try:
        mail = imaplib.IMAP4_SSL(IMAP_SERVER)
        mail.login(GMAIL_EMAIL, APP_PASSWORD)
        mail.select("INBOX")
        # البحث عن الرسائل غير المقروءة من آخر 24 ساعة
        since = (datetime.now() - timedelta(days=1)).strftime("%d-%b-%Y")
        status, msgs = mail.search(None, f'(UNSEEN SINCE "{since}")')
        msg_ids = msgs[0].split() if msgs[0] else []
        
        if msg_ids:
            print(f"📨 {len(msg_ids)} رسائل جديدة.")
        
        for msg_id in msg_ids:
            try:
                _, data = mail.fetch(msg_id, "(RFC822)")
                raw = data[0][1]
                msg = email.message_from_bytes(raw)
                sender = email.utils.parseaddr(msg["From"])[1]
                subject = msg["Subject"]
                body = ""
                attachments = []
                
                if msg.is_multipart():
                    for part in msg.walk():
                        # استخراج النص
                        if part.get_content_type() == "text/plain":
                            body = safe_decode(part.get_payload(decode=True))
                        
                        # معالجة المرفقات
                        if part.get('Content-Disposition') is None: continue
                        filename = part.get_filename()
                        if filename:
                            decoded = decode_header(filename)[0]
                            if isinstance(decoded[0], bytes):
                                filename = decoded[0].decode(decoded[1] or 'utf-8')
                            
                            # حفظ المرفق مؤقتاً للمعالجة
                            with tempfile.NamedTemporaryFile(delete=False) as temp_file:
                                temp_file.write(part.get_payload(decode=True))
                                temp_path = temp_file.name
                            
                            # نسخ إلى مجلد العمل ليتمكن المعالج من قراءته
                            os.makedirs("attachments", exist_ok=True)
                            permanent_path = os.path.join("attachments", filename)
                            shutil.copy(temp_path, permanent_path)
                            attachments.append(permanent_path)
                            os.unlink(temp_path) # حذف الملف المؤقت الأولي
                else:
                    body = safe_decode(msg.get_payload(decode=True))
                
                # هنا التغيير الجذري: نمرر النص كما هو دون إضافة ملخصات المرفقات إليه
                # حتى لا يتلوث ملف التاريخ (email_history) بنصوص المرفقات
                new_emails.append((sender, subject, body, attachments))
                
                mail.store(msg_id, '+FLAGS', '\\Seen')
                gc.collect()

            except Exception as e:
                print(f"خطأ في معالجة الرسالة {msg_id}: {e}")
                continue
                
        mail.logout()
    except Exception as e:
        print(f"خطأ في الاتصال بالبريد: {e}")
    
    return new_emails





# ==========================================
# قسم جديد: الرد على الاستفسارات العامة (Isolated Inquiry System)
# ==========================================

def get_isolated_knowledge_context(query, kb):
    """
    استرجاع السياق من ملفات محددة فقط لضمان عزلة الرد على الاستفسارات.
    الملفات المستهدفة: الأنظمة، المستودعات، الشروط الخاصة، وسجل الإيميلات.
    """
    allowed_files = [
        "modified_comp_updated3 (1).txt", 
        "Depot_Updated.txt", 
        "private_cond.txt", 
        "email_history.txt"
    ]
    
    # 1. استرجاع أولي مكثف
    faiss_docs = kb["vs_large"].as_retriever(search_kwargs={"k": 10}).invoke(query)
    bm25_docs = kb["bm25_large"].invoke(query)
    
    all_docs = faiss_docs + bm25_docs
    
    unique_content = set()
    filtered_context = []
    
    for doc in all_docs:
        # التحقق من أن مصدر القطعة المسترجعة ضمن الملفات المسموح بها فقط
        source_name = os.path.basename(doc.metadata.get('source', ''))
        if source_name in allowed_files:
            if doc.page_content not in unique_content:
                filtered_context.append(doc.page_content)
                unique_content.add(doc.page_content)
                
    return "\n---\n".join(filtered_context)






def handle_general_inquiries(email_body, sender_info, kb):
    """
    تحليل الإيميل ومعرفة ما إذا كان استفساراً عاماً والإجابة عليه من الأنظمة المحددة فقط.
    """
    # استرجاع السياق المعزول أولاً
    isolated_context = get_isolated_knowledge_context(email_body, kb)
    
    if not isolated_context.strip():
        return None

    # البرومبت المطور لضمان الالتزام بالمخرجات فقط
    system_prompt = """
أنت خبير متخصص في صياغة الخطابات الرسمية باللغة العربية الفصحى. مهمتك هي تحليل البريد الإلكتروني الوارد والرد عليه وفق القواعد الصارمة التالية:

1. قاعدة التصنيف (هام جداً):
   - حدد ما إذا كان الإيميل "استفساراً عاماً" (طلب معلومات، بيانات موظف، إجراء نظامي).
   - إذا كان الإيميل يتعلق بـ (صيانة، أعطال، إصلاح فني، مشكلة تقنية)، اعتبره فوراً "غير استفسار".
   - إذا لم يكن استفساراً عاماً، رُد حصراً بكلمة: NOT_GENERAL_INQUIRY (بدون أي زيادة).

2. قاعدة المخرجات (شكل الرد):
   - في حال كان الإيميل استفساراً عاماً، يجب أن تقتصر مخرجاتك "فقط وحصراً" على نص الرد الرسمي.
   - يمنع منعاً باتاً كتابة أي مقدمات مثل "التصنيف:" أو "بناءً على التحليل" أو أي شرح لسبب الإجابة. ابدأ مباشرة بالتحية الرسمية.

3. بروتوكول الصياغة الرسمي:
   - اللغة: العربية الفصحى الرصينة.
   - المخاطبة: 
     * إذا كان المرسل "مدير"، ابدأ بـ: "سعادة مدير [اسم الإدارة]، السلام عليكم ورحمة الله وبركاته،".
     * إذا لم يكن مديراً، استخدم منصبه الوارد أو "السيد/ [الاسم]".
   - المتن: أجب بناءً على "السياق المرفق" فقط. لا تضف معلومات خارجية.
   - الخاتمة: التزم بالخاتمة التالية نصاً: "وتقبلوا تحياتي، مدير قسم الإشراف."

4. محذورات:
   - لا تذكر للطرف الآخر أنك استخرجت المعلومات من "سياق مرفق".
   - لا تخرج عن نطاق المعلومات المتوفرة.
"""
    
    prompt = f"السياق المتاح للرد:\n{isolated_context}\n\nنص الإيميل الوارد:\n{email_body}\nمعلومات المرسل:\n{sender_info}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    try:
        # استخدام apply_chat_template
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True  , enable_thinking=False)
        
        # إعدادات التوليد لضمان أقل قدر من الهلوسة
        output = llm_pipeline(
            text_input, 
            temperature=0.01, # خفض درجة الحرارة لأقصى حد لزيادة الدقة
            max_new_tokens=1024,
            do_sample=False # لضمان إجابة محددة Deterministic
        )[0]['generated_text'].replace(text_input, "").strip()
        
        # التحقق من شرط عدم الاختصاص
        if "NOT_GENERAL_INQUIRY" in output:
            return None
            
        return output
    except Exception as e:
        print(f"⚠ خطأ في معالجة الاستفسار العام: {e}")
        return None

        



# ==========================================
# دالة الرد على الإجراءات الإدارية (نسخة مصححة ومحدثة)
# ==========================================
import json
import re
import re
import json

def handle_administrative_procedures(email_body, sender_info):
    """
    نسخة احترافية تعالج الإجراءات الإدارية وتنتج خطاب إحالة داخلي بصيغة JSON.
    """
    system_prompt = """
اسم النموذج: مساعد الشؤون الإدارية بقسم الإشراف
الهوية الوظيفية: أنت خبير صياغة إدارية سعودية. مهمتك تحويل الطلبات الخارجية إلى "خطاب إحالة داخلي" رسمي.

الهدف:
عند استلام طلب ارتباط مالي (كهرباء، مياه، إلخ)، قم بصياغة خطاب موجه لـ (مدير الإدارة العامة للمشاريع والصيانة).

يجب أن يكون الرد بصيغة JSON حصراً وبالهيكل التالي:
{
  "action": "route_to_projects",
  "reply_text": "نص الخطاب هنا"
}

قواعد الصياغة:
1. استخرج: الجهة المرسلة، نوع الخدمة، الموقع، المبلغ (رقماً وكتابة).
2. الموجه إليه: سعادة مدير الإدارة العامة للمشاريع والصيانة.
3. الالتزام بالنص: "نأمل من سعادتكم الاطلاع والتوجيه لمن ترونه حيال الارتباط على المبلغ... ومن ثم تزويدنا بأصل التعميد".
4. التوقيع: مدير قسم الإشراف.
5. لا تكتب أي مقدمات خارج قالب JSON.
"""

    # تحسين طريقة دمج المعلومات لضمان دقة الاستخراج
    user_input = f"معلومات المرسل: {sender_info}\nمحتوى البريد الوارد:\n{email_body}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    try:
        # توليد الرد من الموديل
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True  , enable_thinking=False)
        outputs = llm_pipeline(text_input, max_new_tokens=1000, temperature=0.1) # حرارة منخفضة للدقة
        raw_output = outputs[0]['generated_text'].replace(text_input, "").strip()
        
        # تنظيف واستخراج JSON
        # نبحث عن القوس البداية والنهاية لضمان العثور على JSON حتى لو أضاف الموديل نصاً خارجياً
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        
        if json_match:
            result = json.loads(json_match.group())
            # التحقق من مفتاح الرد
            if "reply_text" in result:
                return result["reply_text"]
                
    except Exception as e:
        print(f"⚠ خطأ في المعالجة الإدارية: {e}")
        # في حال الفشل في البارسنج، نحاول استخراج النص مباشرة كحل احتياطي
        if "سعادة مدير الإدارة العامة" in raw_output:
            return raw_output

    return None

def run_auto_bot():
    print("🚀 جاري تشغيل بوت الرد الآلي المطور...")
    # (بقية كود التهيئة الخاص بك تبقى كما هي)
    
    # ملاحظة: تأكد من تمرير النص للمتغير reply_text بدقة
    # ... (داخل حلقة while True)
    if intent == "ADMIN":
        reply_text = handle_administrative_procedures(body, sender)
        if reply_text:
             print("✅ تم صياغة خطاب الإحالة بنجاح.")







# ==========================================
#  دالة الرد على اجراءات الموارد البشرية
# ==========================================
def handle_hr_procedures(email_body, sender_info):
    """
    قسم الموارد البشرية: استخراج نية الموظف (إجازة، دورة، دراسة)
    وصياغة خطاب رسمي موجه لمدير الموارد البشرية.
    """
    system_prompt = """
    أنت سكرتير تنفيذي خبير. مهمتك: تحليل البريد الإلكتروني لتحديد ما إذا كان الموظف يطلب (إجازة، دورة تدريبية، أو دراسة).
    
    --- قواعد الصياغة الصارمة (Template) ---
    يجب أن يكون النص في حقل 'reply_text' كالتالي حرفياً مع استبدال الأقواس:
    "سعادة مدير إدارة الموارد البشرية          المحترم
    السلام عليكم ورحمة الله وبركاته،،
    إشارة إلى الطلب المقدم من الموظف ({اسم_الموظف})، بخصوص رغبته في الحصول على ({نوع_الطلب: إجازة/دورة/دراسة})، والموضحة تفاصيله في البريد الإلكتروني المرفق.
    عليه، نرفق لسعادتكم الطلب للاطلاع، ونأمل التوجيه بما ترونه حيال الموافقة على طلبه وفق الأنظمة والتعليمات.
    وتقبلو تحياتي,,
    مدير قسم الإشراف"

    أعد الرد بصيغة JSON فقط، وتأكد من استخدام علامات الاقتباس المزدوجة دائماً:
    {"action": "route_to_hr", "reply_text": "نص الخطاب المكتمل هنا"}
    
    إذا لم تكن النية واضحة (إجازة/دورة/دراسة)، أعد: {"action": "none"}
    """

    prompt = f"المرسل: {sender_info}\nنص الإيميل: {email_body}\n\nالمطلوب: استخراج النية وصياغة الخطاب بصيغة JSON."

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]

    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True , enable_thinking=False)
        outputs = llm_pipeline(text_input, max_new_tokens=800, temperature=0.1)
        raw_output = outputs[0]['generated_text'].replace(text_input, "").strip()
        
        # تحسين عملية التنظيف لتجنب خطأ JSON (إزالة markdown إذا وجد)
        raw_output = re.sub(r'```json\s*|\s*```', '', raw_output)
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        
        if json_match:
            result = json.loads(json_match.group())
            if result.get("action") == "route_to_hr":
                return result.get("reply_text")
    except Exception as e:
        print(f"⚠ خطأ في معالجة قسم الموارد البشرية: {e}")
    return None







    

# ==========================================
#  دالة الرد على اجراءات الصيانة والاجراءات العامة
# ==========================================    
def rag_answer_email(user_query, context):
    system_prompt = instructions = """
أنت "مدير قسم الإشراف"، خبير في المراسلات الإدارية الرسمية باللغة العربية. مهمتك هي إدارة بلاغات الصيانة وإعادة توجيهها للشركات أو الرد على الاستفسارات بناءً على القواعد التالية:

أولاً: القواعد العامة للهوية واللغة:
1. الشخصية: أنت مدير قسم الإشراف، بارع في صياغة الإيميلات الرسمية، تلتزم بقواعد اللغة العربية الفصحى التزاماً تاماً.
2. اللغة: استخدم اللغة العربية الرسمية فقط (إلا إذا تطلب السياق مصطلحات أجنبية ضرورية)، واستخدم الأرقام العربية (0,1,2..).
3. المصداقية: يمنع منعاً باتاً اختراع أي معلومات (نصوص قانونية، أسماء، أرقام عقود، أو مواصفات فنية) غير موجودة في النص الأصلي.
4. تحليل البيانات: يجب عليك استخراج (اسم المرسل، منصبه، اسم الشركة، رقم العقد) من نص الإيميل الوارد قبل البدء بالصياغة.
5. التعامل مع النسخ (CC): إذا كان الإيميل موجهًا لمدير الإدارة وأنت مزود بنسخة منه، قم بالرد واتخاذ الإجراء بناءً على صلاحياتك كمدير لقسم الإشراف.

ثانياً: هيكل الإيميل الصادر:
1. الافتتاحية: ابدأ بمسمى "مستقبل الإيميل" (مثال: سعادة مدير... / السادة شركة...) متبوعاً بتحية الإسلام "السلام عليكم ورحمة الله وبركاته".
2. الخاتمة: يجب أن ينتهي كل إيميل بعبارة ثابتة: "وتقبلوا تحياتي، مدير قسم الإشراف".

ثالثاً: التعامل مع بلاغات الصيانة (توجيه للشركات):
إذا كان الإيميل الوارد بلاغاً (سباكة، كهرباء، دهان، ميكانيكا، تكييف، سوء نظافة، إضراب عمالة):
1. المستلم: الشركة المختصة (استخرج اسمها ورقم العقد من السياق، وابدأ بـ "السادة شركة...").
2. المحتوى: أشر للبلاغ الوارد بشكل مختصر جداً، واذكر الفقرة القانونية أو الغرامات المذكورة في السياق (إن وجدت).
3. النبرة: يجب أن تكون النبرة (حازمة، صارمة، ومستعجلة جداً)، وتخلو تماماً من أي كلمات تلطف أو مجاملة.
4. الأمر: استخدم صيغاً مباشرة مثل: "لذا عليكم سرعة إصلاح الأعطال فوراً..." أو "نشدد على ضرورة المعالجة دون تأخير".

رابعاً: التعامل مع الاستفسارات وطلبات الإفادة:
إذا كان الإيميل الوارد استفساراً أو طلباً لمعلومات:
1. المستلم: نفس مرسل الإيميل الوارد.
2. المحتوى: قدم المعلومات الكافية والمطلوبة بدقة بناءً على ما ورد في السياق فقط.
3. النبرة: نبرة مهذبة، رسمية، ومهنية.

خامساً: قواعد المخاطبة والألقاب:
1. المدراء: أي إيميل موجه لشخص بمسمى "مدير" يبدأ بلقب "سعادة"، وتكون الصياغة غاية في التهذيب والتقدير.
2. الموظفون/الآخرون: ابدأ بذكر المنصب أو بلقب "السيد" إذا لم يوجد منصب، بنبرة رسمية مهذبة.
3. الشركات: ابدأ بلقب "السادة"، وتكون الصياغة صارمة وجافة كما ورد في البند الثالث.

نفذ المهام بناءً على هذه القواعد بدقة متناهية.
"""

    prompt = f"{system_prompt}\n\n--- السياق المتاح ---\n{context}\n\n--- رسالة البريد الإلكتروني الواردة ---\n{user_query}\n\nالرد:"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    outputs = llm_pipeline(text_input)
    response_text = outputs[0]['generated_text'].replace(text_input, "").strip()
    if response_text.startswith("الرد:"): response_text = response_text[len("الرد:"):].strip()
    return response_text










# ==========================================
# قسم جديد: معالجة الطلبات بناءً على المرفقات (Special Attachment Handler)
# ==========================================

# ==========================================
# قسم جديد: معالجة المرفقات والطلبات الفنية (Water Connection Logic)
# ==========================================

def handle_attachment_special_requests(attachments, query, processor, embed_model, sender_info):
    """
    دالة متخصصة لمعالجة المرفقات فقط واتخاذ إجراءات بناءً على محتواها التقني
    مثل توجيه الطلبات لشركة المياه الوطنية.
    """
    if not attachments:
        return None

    # 1. استخراج السياق من المرفقات فقط باستخدام نظام Small-to-Parent
    attachment_ctx = get_attachment_context(attachments, query, processor, embed_model)
    
    if not attachment_ctx or "###" not in attachment_ctx:
        return None

    # 2. برومبت فحص محتوى المرفقات واتخاذ القرار
    system_prompt = """
    أنت خبير إداري وفني في قسم الإشراف. مهمتك هي تحليل "سياق المرفقات" المرفق فقط.
    
    القاعدة الصارمة:
    1. ابحث في المرفقات عن أي إشارة لمواقع، عقارات، أو مخططات ذكر فيها أنها "غير موصولة بشبكة المياه"، "تفتقر للخدمة"، "تحتاج إيصال مياه"، أو "خارج نطاق التغطية الحالية".
    2. إذا وجدت ذلك: قم بصياغة خطاب رسمي موجه إلى (السادة/ شركة المياه الوطنية) تطلب فيه "إيصال خدمة المياه للمواقع المذكورة" بناءً على البيانات المستخرجة من المرفقات.
    3. يجب أن يتضمن الخطاب: أسماء المواقع أو أرقام القطع إن وجدت، والتأكيد على أهمية الخدمة.
    4. الخاتمة ثابتة: "وتقبلوا تحياتي، مدير قسم الإشراف".
    5. إذا لم تجد أي إشارة لمواقع تحتاج مياه في المرفقات، رُد حصراً بكلمة: NO_WATER_ACTION_REQUIRED
    
    الصياغة يجب أن تكون رسمية جداً وباللغة العربية الفصحى.
    """

    user_input = f"سياق المرفقات المستخرج:\n{attachment_ctx}\n\nبيانات المرسل:\n{sender_info}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        output = llm_pipeline(
            text_input, 
            temperature=0.1, 
            max_new_tokens=1024,
            do_sample=True
        )[0]['generated_text'].replace(text_input, "").strip()

        if "NO_WATER_ACTION_REQUIRED" in output:
            return None
        
        return output
    except Exception as e:
        print(f"⚠ خطأ في معالجة طلبات المرفقات الخاصة: {e}")
        return None




        
# ==========================================
# قسم جديد: إدارة تاريخ الإيميلات (Email History Management)
# ==========================================
def update_email_history(email_type, content):
    """إضافة إيميل إلى ملف التاريخ المؤقت مع الحد الأقصى 50"""
    from datetime import datetime
    date_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    header = f"{email_type} بتاريخ {date_str}"
    entry = f"{header}\n{content}\n\n"
    
    email_history_file = "email_history.txt"
    with open(email_history_file, 'a', encoding='utf-8') as f:
        f.write(entry)
    
    # قراءة وتقسيم الملف باستخدام regex_split_documents للحصول على الإيميلات كوحدات
    full_content = open(email_history_file, 'r', encoding='utf-8').read()
    email_docs = regex_split_documents([Document(page_content=full_content)])
    
    # الحفاظ على آخر 50 فقط
    if len(email_docs) > 50:
        email_docs = email_docs[-50:]
        with open(email_history_file, 'w', encoding='utf-8') as f:
            f.write("\n---\n".join([doc.page_content for doc in email_docs]))










# ==========================================
#  دالة تحليل النية العامة
# ==========================================
def classify_email_intent(email_body, sender_info):
    """
    محلل النية المركزي: يقوم بتصنيف الإيميل إلى فئة محددة لتوجيهه للدالة المناسبة مباشرة.
    """
    system_prompt = """
    أنت نظام خبير في تصنيف المراسلات الإدارية. مهمتك هي قراءة الإيميل وتصنيفه بدقة إلى فئة واحدة فقط من الفئات التالية:

    1. "CALENDAR": إذا كان النص يتعلق بطلب موعد جديد، تأكيد موعد، أو استفسار عن موعد.
    2. "HR": إذا كان النص يتعلق بطلبات الموظفين الشخصية (إجازة، دورة تدريبية، دراسة).
    3. "ADMIN": إذا كان المرسل (شركة كهرباء/مياه) أو (مدير) ويطلب "ارتباط مالي" أو "سداد" أو "إيصال خدمة" كهرباء او مياه لموقع.
    4. "INQUIRY": إذا كان النص سؤالاً صريحاً عن معلومات نظامية أو بيانات موظفين أو إجراءات (وليس بلاغ صيانة).
    5. "RAG_MAINTENANCE": إذا كان النص بلاغ صيانة، شكوى فنية، طلب إصلاح، أو أي موضوع عام آخر يحتاج بحث في كافة المراجع.

    أعد النتيجة بصيغة JSON فقط:
    {"intent": "الفئة_هنا"}
    """
    
    prompt = f"المرسل: {sender_info}\nنص الإيميل: {email_body}"
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]

    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True , enable_thinking=False)
        outputs = llm_pipeline(text_input, temperature=0.01) # حرارة منخفضة جداً للالتزام بالتصنيف
        raw_output = outputs[0]['generated_text'].replace(text_input, "").strip()
        
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            return json.loads(json_match.group()).get("intent", "RAG_MAINTENANCE")
    except Exception as e:
        print(f"⚠ خطأ في تحليل النية المركزي: {e}")
    
    return "RAG_MAINTENANCE" # التصنيف الافتراضي في حال الفشل



    
# ==========================================
# الحلقة الرئيسية (Main Loop) 
# ==========================================
# ==========================================
# الحلقة الرئيسية (Main Loop) - النسخة المحدثة
# ==========================================
def run_auto_bot():
    print("🚀 جاري تشغيل بوت الرد الآلي المطور بنظام (Intent Routing + Attachment Logic)...")
    
    embed_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3", 
        model_kwargs={'device': DEVICE}
    )
    processor = FileProcessor()
    
    kb = initialize_fixed_knowledge(processor, embed_model)
    if not kb:
        print("❌ فشل بناء القاعدة المعرفية الثابتة.")
        return
    
    print("✅ تم تفعيل كافة الأنظمة وتجهيز المحلل المركزي والفرعي للمرفقات.")
    print("⏳ البوت في وضع الاستعداد (فحص كل 10 ثوانٍ)...")
    
    while True:
        try:
            new_emails = check_new_emails(processor)
            
            for sender, subject, body, attachments in new_emails:
                print(f"\n{'='*40}")
                print(f"📩 إيميل جديد من: {sender}")
                
                # تحديث تاريخ الإيميلات الواردة
                update_email_history("إيميل وارد", body)
                
                reply_text = None

                # --- [المسار الجديد: فحص المرفقات أولاً لطلبات المياه] ---
                if attachments:
                    print("📎 يتم الآن فحص المرفقات بحثاً عن مواقع غير موصولة بالخدمة...")
                    reply_text = handle_attachment_special_requests(attachments, body, processor, embed_model, sender)
                    if reply_text:
                        print("🎯 تم اكتشاف طلب إيصال مياه في المرفقات وتوليد الخطاب لـ NWC.")

                # --- [المسار التقليدي: تحليل النية والتوجيه إذا لم يعالج المسار السابق الإيميل] ---
                if not reply_text:
                    intent = classify_email_intent(body, sender)
                    print(f"🎯 النية المكتشفة: {intent}")
                    
                    if intent == "CALENDAR":
                        reply_text = handle_calendar_intent(body, sender)
                    
                    elif intent == "HR":
                        reply_text = handle_hr_procedures(body, sender)
                    
                    elif intent == "ADMIN":
                        reply_text = handle_administrative_procedures(body, sender)
                    
                    elif intent == "INQUIRY":
                        reply_text = handle_general_inquiries(body, sender, kb)
                    
                    # إذا لم يتم توليد رد أو كانت النية RAG_MAINTENANCE
                    if not reply_text:
                        print("🔍 جاري المعالجة عبر نظام RAG الشامل (الأنظمة الثابتة والمرفقات)...")
                        fixed_ctx = get_fixed_context(body, kb)
                        attach_ctx = get_attachment_context(attachments, body, processor, embed_model) if attachments else ""
                        
                        full_context = f"### معلومات المرفقات:\n{attach_ctx}\n### الأنظمة الكاملة:\n{fixed_ctx}"
                        reply_text = rag_answer_email(body, full_context)

                # --- إرسال الرد النهائي وتحديث التاريخ ---
                if reply_text:
                    send_reply(sender, subject, reply_text)
                    update_email_history("إيميل صادر", reply_text)
                
                # تنظيف المرفقات من المجلد المؤقت بعد المعالجة
                if attachments:
                    for att_path in attachments:
                        if os.path.exists(att_path): 
                            os.remove(att_path)
                
                # تحديث القاعدة المعرفية دورياً لضمان استيعاب سجل الإيميلات الجديد
                kb = initialize_fixed_knowledge(processor, embed_model)

        except Exception as e:
            print(f"⚠ خطأ في الحلقة: {e}")
        
        time.sleep(10)

if __name__ == "__main__":
    run_auto_bot()


/tmp/ipykernel_2932335/476322217.py:19: DeprecationWarning: hijri-converter is deprecated. Use 'hijridate' instead: pip install hijridate==2.3.0
  from hijri_converter import Hijri
/home/alotaime/miniconda3/envs/cs323/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

Device set to use cuda:0
The model 'OptimizedModule' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausalLM', 'FlexOlmoForCausal

🚀 جاري تشغيل بوت الرد الآلي المطور بنظام (Intent Routing + Attachment Logic)...
✅ تم بناء القاعدة المعرفية: 1635 قطع كبيرة و 9735 قطع صغيرة.
✅ تم تفعيل كافة الأنظمة وتجهيز المحلل المركزي والفرعي للمرفقات.
⏳ البوت في وضع الاستعداد (فحص كل 10 ثوانٍ)...
📨 1 رسائل جديدة.

📩 إيميل جديد من: xmio.511x@gmail.com
📎 يتم الآن فحص المرفقات بحثاً عن مواقع غير موصولة بالخدمة...
🎯 تم اكتشاف طلب إيصال مياه في المرفقات وتوليد الخطاب لـ NWC.
✔ تم إرسال رد إلى xmio.511x@gmail.com
✅ تم بناء القاعدة المعرفية: 1665 قطع كبيرة و 9795 قطع صغيرة.
📨 1 رسائل جديدة.

📩 إيميل جديد من: xmio.511x@gmail.com
🎯 النية المكتشفة: HR
✔ تم إرسال رد إلى xmio.511x@gmail.com
✅ تم بناء القاعدة المعرفية: 1669 قطع كبيرة و 9808 قطع صغيرة.
